In [1]:
%load_ext autoreload
%autoreload 2
import json
from dotenv import load_dotenv
import json
import os

In [ ]:
load_dotenv()

GIRDER_API_URL = os.getenv("GIRDER_API_URL")
API_KEY = os.getenv("HTMDEC_GIRDER_API_KEY")
ALPSS_FORM_ID = '69b2778b7590bb7e1ed158a1'
PDV_FOLDER_ID = '69c3ef40c855e9e387f24425'

In [3]:
from girder_client import GirderClient
client = GirderClient(apiUrl=GIRDER_API_URL)
client.authenticate(apiKey=API_KEY)
user = client.get("user/me")
print(f"✅ Authenticated as {user['login']}")

HttpError: HTTP error 400: POST https://data.htmdec.org/api/v1/api_key/token?key=zFzzuZyhIb1cwp7eeYAg86ek6k2fLIDDRtSQGCcz
Response text: {"message": "Invalid API key.", "type": "validation"}

In [17]:
igsns = client.get('deposition', parameters={'limit': 10000,  'sort': 'created', 'sortdir': -1})
# igsns = igsns[:30]
igsns = [{"igsn": 'JHAMAC00003-S1R4C3'}, {"igsn": 'JHAMAC00003-S4R5C3'}, {"igsn": 'JHAMAC00003-S1R6C3'}, {"igsn": 'JHAMAC00001-S8R3C2'}]
# EXLUDE UNNECESSARY IGSN LIKE TAMU
print("number of igsns: ", len(igsns))

number of igsns:  4


In [ ]:
def analyze_xrd_files_for_igsn(client, igsn):
    params = {
        "q": igsn,
        "mode": "igsn",
        "types": '["folder","item"]',
        "limit": 1000,
    }

    resources = client.get("resource/search", parameters=params)

    xrd_files_count = 0
    ebsd_files_count = 0

    for resource_type_list in resources.values():
        for resource in resource_type_list:
            name = resource.get("name", "")
            if name.endswith("_xrd.csv"):
                xrd_files_count += 1

    return {
        "has_xrd": xrd_files_count > 0,
        "xrd_files_count": xrd_files_count,
        "has_ebsd": ebsd_files_count > 0,
        "ebsd_files_count": ebsd_files_count,
    }


def analyze_laser_shock_files_for_igsn(client, igsn):
    """Detect laser shock presence and shot count from files inside the PDV folder whose name contains the IGSN."""
    items = client.get("item", parameters={
        "folderId": PDV_FOLDER_ID,
        "limit": 100000,
    })

    shots = [item for item in items if igsn in item.get("name", "")]
    count = len(shots)
    return {
        "has_laser_shock": count > 0,
        "laser_shock_shots_count": count,
    }


def analyze_laser_shock_forms_for_igsn(client, igsn):
    """Get quality flag counts (velocity_ok, spall_ok, hel_detected) from ALPSS form entries."""
    results = client.get(
        "entry",
        parameters={
            "formId": ALPSS_FORM_ID,
            "query": igsn,
            "field": "Sample_IGSN",
            "limit": 100000,
        },
    )

    if not results:
        return {
            "velocity_ok_count": 0,
            "spall_ok_count": 0,
            "hel_detected_count": 0,
        }

    return {
        "velocity_ok_count": sum(1 for e in results if e.get("data", {}).get("velocity_ok") is True),
        "spall_ok_count":    sum(1 for e in results if e.get("data", {}).get("spall_ok")    is True),
        "hel_detected_count": sum(1 for e in results if e.get("data", {}).get("hel_detected") is True),
    }

In [ ]:
from tqdm import tqdm

analysis_results = {}

for dep in tqdm(igsns, desc="Analyzing IGSNs"):
    igsn = dep["igsn"]

    file_info    = analyze_xrd_files_for_igsn(client, igsn)
    shock_files  = analyze_laser_shock_files_for_igsn(client, igsn)
    shock_forms  = analyze_laser_shock_forms_for_igsn(client, igsn)

    has_any_data = (
        file_info["has_xrd"]
        or file_info["has_ebsd"]
        or shock_files["has_laser_shock"]
    )

    analysis_results[igsn] = {
        "has_any_data": has_any_data,
        **file_info,
        **shock_files,
        **shock_forms,
    }

In [21]:
summary = {
    "total_igsns": len(analysis_results),
    "igsns_with_any_data": 0,
    "igsns_with_xrd": 0,
    "igsns_with_ebsd": 0,
    "igsns_with_laser_shock": 0,
    "total_xrd_files": 0,
    "total_ebsd_files": 0,
    "total_laser_shock_entries": 0,
}

for data in analysis_results.values():
    if data["has_any_data"]:
        summary["igsns_with_any_data"] += 1
    if data["has_xrd"]:
        summary["igsns_with_xrd"] += 1
    if data["has_ebsd"]:
        summary["igsns_with_ebsd"] += 1
    if data["has_laser_shock"]:
        summary["igsns_with_laser_shock"] += 1

    summary["total_xrd_files"] += data["xrd_files_count"]
    summary["total_ebsd_files"] += data["ebsd_files_count"]
    summary["total_laser_shock_entries"] += data["laser_shock_shots_count"]

print(json.dumps(summary, indent=3))


{
   "total_igsns": 4,
   "igsns_with_any_data": 4,
   "igsns_with_xrd": 2,
   "igsns_with_ebsd": 0,
   "igsns_with_laser_shock": 4,
   "total_xrd_files": 180,
   "total_ebsd_files": 0,
   "total_laser_shock_entries": 91
}


In [8]:
import pandas as pd
df = (
    pd.DataFrame.from_dict(analysis_results, orient="index")
    .reset_index()
    .rename(columns={"index": "igsn"})
)

df = df.sort_values(
    by=["has_xrd", "has_laser_shock"],
    ascending=[False, False],
).reset_index(drop=True)

df.head()

,igsn,has_any_data,has_xrd,xrd_files_count,has_ebsd,ebsd_files_count,has_laser_shock,laser_shock_shots_count,velocity_ok_count,spall_ok_count,hel_detected_count
0,JHAMAC00003-S1R4C3,True,True,139,False,0,True,25,25,22,0
1,JHAMAC00003-S1R6C3,True,True,41,False,0,True,25,25,7,0
2,JHAMAC00003-S4R5C3,True,False,0,False,0,True,25,25,25,0
3,JHAMAC00001-S8R3C2,True,False,0,False,0,True,16,16,8,1


In [9]:
# 1) Overlap / Venn table
xrd_only     = df[df["has_xrd"] & ~df["has_laser_shock"]]
shock_only   = df[~df["has_xrd"] & df["has_laser_shock"]]
both         = df[df["has_xrd"] & df["has_laser_shock"]]
neither      = df[~df["has_xrd"] & ~df["has_laser_shock"]]
both_spall   = df[df["has_xrd"] & df["has_laser_shock"] & df["spall_ok_count"].gt(0)]
both_hel     = df[df["has_xrd"] & df["has_laser_shock"] & df["hel_detected_count"].gt(0)]
both_spall_hel = df[df["has_xrd"] & df["has_laser_shock"] & df["spall_ok_count"].gt(0) & df["hel_detected_count"].gt(0)]

categories = [
    ("XRD only",                          xrd_only),
    ("Laser shock only",                  shock_only),
    ("Both XRD & laser shock",            both),
    ("Neither",                           neither),
    ("Both XRD & laser shock & spall",    both_spall),
    ("Both XRD & laser shock & HEL",      both_hel),
    ("Both XRD & laser shock & spall & HEL", both_spall_hel),
]

overlap = pd.DataFrame({
    "category": [c for c, _ in categories],
    "count":    [len(d) for _, d in categories],
    "pct":      [round(100*len(d)/len(df), 1) for _, d in categories],
})
print(overlap.to_string(index=False))

                            category  count  pct
                            XRD only      0  0.0
                    Laser shock only      2 50.0
              Both XRD & laser shock      2 50.0
                             Neither      0  0.0
      Both XRD & laser shock & spall      2 50.0
        Both XRD & laser shock & HEL      0  0.0
Both XRD & laser shock & spall & HEL      0  0.0


In [10]:
# TODO: location specific analysis

In [11]:
# 2) Quality flags (velocity_ok, spall_ok, hel_detected) — laser shock IGSNs only
shock_df = df[df["has_laser_shock"]].copy()
total_shock = len(shock_df)

quality = pd.DataFrame({
    "flag":    ["velocity_ok", "spall_ok", "hel_detected"],
    "igsns":   [
        shock_df["velocity_ok_count"].gt(0).sum(),
        shock_df["spall_ok_count"].gt(0).sum(),
        shock_df["hel_detected_count"].gt(0).sum(),
    ],
    "total_shots": [
        shock_df["velocity_ok_count"].sum(),
        shock_df["spall_ok_count"].sum(),
        shock_df["hel_detected_count"].sum(),
    ],
    "total_laser_shots": summary["total_laser_shock_entries"],
})
quality["shot_pct"] = (quality["total_shots"] / quality["total_laser_shots"] * 100).round(1)
print(quality.to_string(index=False))

        flag  igsns  total_shots  total_laser_shots  shot_pct
 velocity_ok      4           91                 91     100.0
    spall_ok      4           62                 91      68.1
hel_detected      1            1                 91       1.1


In [12]:
# 4) Gap analysis
print("=== XRD but no laser shock ===")
print(xrd_only[["igsn", "xrd_files_count"]].to_string(index=False) if len(xrd_only) else "  (none)")

print("\n=== Laser shock but no XRD ===")
print(shock_only[["igsn", "laser_shock_shots_count"]].to_string(index=False) if len(shock_only) else "  (none)")

=== XRD but no laser shock ===
  (none)

=== Laser shock but no XRD ===
              igsn  laser_shock_shots_count
JHAMAC00003-S4R5C3                       25
JHAMAC00001-S8R3C2                       16


In [13]:
# Dump df to file
df.to_csv("igsn_analysis.csv", index=False)
print("Saved to igsn_analysis.csv")

Saved to igsn_analysis.csv
